In [1]:
from pathlib import Path
import json
import random
import hashlib
import time
from bisect import bisect_left
from collections import Counter
from dataclasses import replace
from traceback import format_exception_only

import pandas as pd
from tqdm.auto import tqdm

In [2]:
PROJECT_ROOT = Path.cwd()

DATA_ROOT = PROJECT_ROOT / "data" / "raw"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("src exists:", (PROJECT_ROOT / "src").exists())
print("data exists:", DATA_ROOT.exists())

PROJECT_ROOT: /home/vios/PycharmProjects/serialization-strategies
DATA_ROOT: /home/vios/PycharmProjects/serialization-strategies/data/raw
src exists: True
data exists: True


In [3]:
from src.dataset_loaders import (
    FCCInvoiceLoader,
    NDALoader,
    SECS1Loader,
    CharityReportLoader,
    ResourceContractLoader,
)

from src.preprocessing.label_utils import label_to_key

from src.preprocessing.value_normalization import (
    loose_value_match,
    normalize_for_matching,
)

from src.serialization import (
    IGNORE_LABEL,
    PlainTextSerializer,
    PageAwareSerializer,
    BlockAwareSerializer,
    LineAwareSerializer,
    RowColBucketSerializer,
    BBoxTokenSerializer,
    ColumnAwareSerializer,
    XYCutAwareSerializer,
    LMDXCoordSuffixSerializer,
    CompactBBoxTokenSerializer,
    TOKEN_CLASSIFICATION_SERIALIZERS,
    collapse_serialized_labels_to_original,
)

In [4]:
required_serializers = {
    "plain_text",
    "page_aware",
    "block_aware",
    "line_aware",
    "rowcol_bucket",
    # "bbox_token",
    "column_aware",
    "xycut_aware",
    "lmdx_coord_suffix",
    "compact_bbox_token",
}

missing = required_serializers.difference(TOKEN_CLASSIFICATION_SERIALIZERS)


In [5]:
SPLIT_SEED = 42

TRAIN_RATIO = 0.80
VAL_RATIO = 0.10
TEST_RATIO = 0.10

RUN_MODE = "full"  # "smoke" or "full"

SMOKE_DOCS_PER_DATASET = 25
MAX_DOCS_PER_DATASET = SMOKE_DOCS_PER_DATASET if RUN_MODE == "smoke" else None

# For debugging keep this False. Add rowcol_bucket/bbox_token later as ablations.
INCLUDE_HEAVY_SERIALIZERS = False
HEAVY_SERIALIZERS = {"bbox_token"}

# Validation is sampled by default.
VALIDATE_EVERY_RECORD = False
VALIDATE_FIRST_N_DOCS_PER_DATASET = 5

# Quality diagnostics are sampled by default.
ASSESS_QUALITY_EVERY_RECORD = False
ASSESS_QUALITY_FIRST_N_DOCS_PER_DATASET = 10

# Writing is streaming, not in-memory.
WRITE_PROCESSED = True
PROCESSED_ROOT = DATA_ROOT / "processed"


USE_INLINE_OCR_COLUMN = True
MIN_TOKEN_OVERLAP = 0.5

print({
    "RUN_MODE": RUN_MODE,
    "MAX_DOCS_PER_DATASET": MAX_DOCS_PER_DATASET,
    "INCLUDE_HEAVY_SERIALIZERS": INCLUDE_HEAVY_SERIALIZERS,
    "VALIDATE_EVERY_RECORD": VALIDATE_EVERY_RECORD,
    "ASSESS_QUALITY_EVERY_RECORD": ASSESS_QUALITY_EVERY_RECORD,
    "WRITE_PROCESSED": WRITE_PROCESSED,
})

{'RUN_MODE': 'full', 'MAX_DOCS_PER_DATASET': None, 'INCLUDE_HEAVY_SERIALIZERS': False, 'VALIDATE_EVERY_RECORD': False, 'ASSESS_QUALITY_EVERY_RECORD': False, 'WRITE_PROCESSED': True}


In [6]:
DATASET_REGISTRY = {
    # "fcc_invoices": {
    #     "folder": "fcc_invoices",
    #     "loader_cls": FCCInvoiceLoader,
    # },
    # "ndas": {
    #     "folder": "nda",
    #     "loader_cls": NDALoader,
    # },
    # "charity_reports": {
    #     "folder": "charities",
    #     "loader_cls": CharityReportLoader,
    # },
    "resource_contracts": {
        "folder": "resource_contracts",
        "loader_cls": ResourceContractLoader,
    },
}

SOURCE_SPLITS = ["train", "val", "test"]

In [7]:
def load_csv_split(dataset_name: str, folder: str, split_name: str) -> pd.DataFrame:
    path = DATA_ROOT / folder / f"{split_name}.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing split file: {path}")

    df = pd.read_csv(path)
    df = df.copy()
    df["dataset"] = dataset_name
    df["source_folder"] = folder
    df["source_split"] = split_name
    df["source_row_index"] = list(range(len(df)))
    return df

combined_tables = {}
split_load_rows = []

for dataset_name, spec in DATASET_REGISTRY.items():
    split_dfs = []

    for split_name in SOURCE_SPLITS:
        try:
            split_df = load_csv_split(dataset_name, spec["folder"], split_name)
            split_dfs.append(split_df)
            split_load_rows.append({
                "dataset": dataset_name,
                "folder": spec["folder"],
                "source_split": split_name,
                "status": "ok",
                "rows": len(split_df),
            })
        except Exception as e:
            split_load_rows.append({
                "dataset": dataset_name,
                "folder": spec["folder"],
                "source_split": split_name,
                "status": f"error: {type(e).__name__}: {e}",
                "rows": None,
            })

    if split_dfs:
        combined = pd.concat(split_dfs, ignore_index=True)

        if MAX_DOCS_PER_DATASET is not None:
            combined = combined.sample(
                n=min(MAX_DOCS_PER_DATASET, len(combined)),
                random_state=SPLIT_SEED,
            ).reset_index(drop=True)

        combined["combined_row_index"] = list(range(len(combined)))
        combined_tables[dataset_name] = combined

split_load_df = pd.DataFrame(split_load_rows)
display(split_load_df)

combined_summary_df = pd.DataFrame([
    {
        "dataset": dataset_name,
        "n_rows": len(df),
        "source_split_counts": dict(df["source_split"].value_counts()),
    }
    for dataset_name, df in combined_tables.items()
])
display(combined_summary_df)

assert combined_tables, "No datasets loaded."

,dataset,folder,source_split,status,rows
0,resource_contracts,resource_contracts,train,ok,117
1,resource_contracts,resource_contracts,val,ok,41
2,resource_contracts,resource_contracts,test,ok,40


,dataset,n_rows,source_split_counts
0,resource_contracts,198,"{'train': 117, 'val': 41, 'test': 40}"


In [8]:
def stable_dataset_seed(global_seed: int, dataset_name: str) -> int:
    digest = hashlib.md5(f"{global_seed}:{dataset_name}".encode("utf-8")).hexdigest()
    return int(digest[:8], 16)

def make_doc_key(dataset_name: str, row: pd.Series) -> str:
    doc_id = str(row.get("original_filename", row.get("doc_id", row.get("id", ""))))
    return (
        f"{dataset_name}"
        f"::source_split={row['source_split']}"
        f"::source_row_index={row['source_row_index']}"
        f"::doc_id={doc_id}"
    )

def assign_seeded_splits_for_dataset(
    dataset_name: str,
    doc_keys: list[str],
    seed: int,
    train_ratio: float,
    val_ratio: float,
    test_ratio: float,
) -> dict[str, str]:
    unique_keys = sorted(set(doc_keys))
    rng = random.Random(stable_dataset_seed(seed, dataset_name))
    rng.shuffle(unique_keys)

    n = len(unique_keys)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)

    train_keys = set(unique_keys[:n_train])
    val_keys = set(unique_keys[n_train:n_train + n_val])
    test_keys = set(unique_keys[n_train + n_val:])

    assignment = {}
    for key in train_keys:
        assignment[key] = "train"
    for key in val_keys:
        assignment[key] = "val"
    for key in test_keys:
        assignment[key] = "test"

    return assignment

split_assignments = {}
assignment_rows = []

for dataset_name, df in combined_tables.items():
    doc_keys = [make_doc_key(dataset_name, row) for _, row in df.iterrows()]
    assignment = assign_seeded_splits_for_dataset(
        dataset_name=dataset_name,
        doc_keys=doc_keys,
        seed=SPLIT_SEED,
        train_ratio=TRAIN_RATIO,
        val_ratio=VAL_RATIO,
        test_ratio=TEST_RATIO,
    )
    split_assignments[dataset_name] = assignment

    counts = Counter(assignment.values())
    assignment_rows.append({
        "dataset": dataset_name,
        "n_docs": len(assignment),
        "train": counts.get("train", 0),
        "val": counts.get("val", 0),
        "test": counts.get("test", 0),
    })

assignment_df = pd.DataFrame(assignment_rows)
display(assignment_df)

,dataset,n_docs,train,val,test
0,resource_contracts,198,158,19,21


In [9]:
loaders = {
    dataset_name: spec["loader_cls"](
        data_root=DATA_ROOT,
        keep_raw_ocr=False,
        strict=False,
    )
    for dataset_name, spec in DATASET_REGISTRY.items()
    if dataset_name in combined_tables
}

all_serializers = {
    "plain_text": PlainTextSerializer(),
    "page_aware": PageAwareSerializer(),
    "block_aware": BlockAwareSerializer(),
    "line_aware": LineAwareSerializer(),

    "column_aware": ColumnAwareSerializer(
        min_gap_ratio=0.035,
        min_tokens_per_column=8,
        max_columns=4,
    ),
    "xycut_aware": XYCutAwareSerializer(
        min_gap_ratio_x=0.06,
        min_gap_ratio_y=0.035,
        min_tokens_per_region=8,
        max_depth=6,
    ),
    "lmdx_coord_suffix": LMDXCoordSuffixSerializer(
        n_buckets=100,
        coord_mode="center",
    ),
    "compact_bbox_token": CompactBBoxTokenSerializer(
        n_buckets=100,
        position="prefix",
    ),

    # Heavy ablations
    "rowcol_bucket": RowColBucketSerializer(n_buckets=100),
    "bbox_token": BBoxTokenSerializer(n_buckets=100),
}

if INCLUDE_HEAVY_SERIALIZERS:
    serializers = all_serializers
else:
    serializers = {
        name: serializer
        for name, serializer in all_serializers.items()
        if name not in HEAVY_SERIALIZERS
    }

print("Datasets:", sorted(loaders))
print("Serializers:", sorted(serializers))

Datasets: ['resource_contracts']
Serializers: ['block_aware', 'column_aware', 'compact_bbox_token', 'line_aware', 'lmdx_coord_suffix', 'page_aware', 'plain_text', 'rowcol_bucket', 'xycut_aware']


In [10]:
def char_overlap(a_start: int, a_end: int, b_start: int, b_end: int) -> int:
    return max(0, min(a_end, b_end) - max(a_start, b_start))

def doc_with_updates(doc, **updates):
    if hasattr(doc, "with_updates"):
        return doc.with_updates(**updates)
    return replace(doc, **updates)

def build_token_offset_index(tokens):
    starts = [int(t.start) for t in tokens]
    sorted_by_start = all(starts[i] <= starts[i + 1] for i in range(len(starts) - 1))
    return {
        "starts": starts,
        "sorted_by_start": sorted_by_start,
    }

def fast_tokens_for_annotation(tokens, annotation, token_index=None, min_token_overlap: float = 0.5):
    """Fast span-to-token alignment.

    Uses a small candidate window around annotation offsets when tokens are sorted
    by document offset. Falls back to a full scan when sorting is not guaranteed.
    """
    if not tokens:
        return []

    ann_start = int(annotation.start)
    ann_end = int(annotation.end)

    if token_index is not None and token_index.get("sorted_by_start", False):
        starts = token_index["starts"]

        # Include a small backtrack because annotations can start inside a token.
        lo = max(0, bisect_left(starts, ann_start) - 8)

        # Include a small lookahead because tokens near span end may start just
        # before or after normalized offsets in noisy OCR streams.
        hi = min(len(tokens), bisect_left(starts, ann_end) + 8)
        candidate_indices = range(lo, hi)
    else:
        candidate_indices = range(len(tokens))

    matched = []

    for i in candidate_indices:
        token = tokens[i]

        overlap = char_overlap(
            int(token.start),
            int(token.end),
            ann_start,
            ann_end,
        )

        if overlap <= 0:
            continue

        ratio = overlap / max(1, int(token.end) - int(token.start))
        if ratio >= min_token_overlap:
            matched.append(i)

    return matched

def fast_reconstruct_token_text(tokens, indices):
    return " ".join(tokens[i].text for i in indices).strip()

def fast_annotation_check(doc, annotation, token_index):
    indices = fast_tokens_for_annotation(
        doc.tokens,
        annotation,
        token_index=token_index,
        min_token_overlap=MIN_TOKEN_OVERLAP,
    )

    token_text = fast_reconstruct_token_text(doc.tokens, indices)
    token_available = len(indices) > 0
    strict_match = " ".join(token_text.split()) == " ".join(str(annotation.text).split())
    loose_match = loose_value_match(annotation.label, annotation.text, token_text)

    return {
        "label": annotation.label,
        "annotation_text": annotation.text,
        "token_text": token_text,
        "token_indices": indices,
        "token_available": token_available,
        "strict_match": strict_match,
        "loose_value_match": loose_match,
        "annotation_norm": normalize_for_matching(annotation.label, annotation.text),
        "token_norm": normalize_for_matching(annotation.label, token_text),
    }

def fast_quality_summary(doc, token_index):
    checks = [
        fast_annotation_check(doc, ann, token_index)
        for ann in (doc.annotations or [])
    ]

    if not checks:
        return {
            "token_available_rate": 1.0,
            "strict_match_rate": 1.0,
            "loose_value_match_rate": 1.0,
            "n_unavailable": 0,
            "n_loose_mismatch": 0,
        }

    return {
        "token_available_rate": sum(c["token_available"] for c in checks) / len(checks),
        "strict_match_rate": sum(c["strict_match"] for c in checks) / len(checks),
        "loose_value_match_rate": sum(c["loose_value_match"] for c in checks) / len(checks),
        "n_unavailable": sum(1 for c in checks if not c["token_available"]),
        "n_loose_mismatch": sum(1 for c in checks if c["token_available"] and not c["loose_value_match"]),
    }

def fast_filter_annotations_for_training(
    doc,
    token_index,
    drop_unavailable: bool = True,
    drop_loose_mismatch: bool = False,
    attach_quality_report: bool = False,
):
    if doc.annotations is None:
        return doc

    kept = []
    checks = [] if attach_quality_report else None

    for ann in doc.annotations:
        check = fast_annotation_check(doc, ann, token_index)
        if checks is not None:
            checks.append(check)

        if drop_unavailable and not check["token_available"]:
            continue
        if drop_loose_mismatch and check["token_available"] and not check["loose_value_match"]:
            continue

        kept.append(ann)

    metadata = dict(doc.metadata)

    if attach_quality_report and checks is not None:
        n = len(checks)
        metadata["quality_report_fast"] = {
            "n_annotations": n,
            "token_available_rate": 1.0 if n == 0 else sum(c["token_available"] for c in checks) / n,
            "strict_match_rate": 1.0 if n == 0 else sum(c["strict_match"] for c in checks) / n,
            "loose_value_match_rate": 1.0 if n == 0 else sum(c["loose_value_match"] for c in checks) / n,
            "n_unavailable": sum(1 for c in checks if not c["token_available"]),
            "n_loose_mismatch": sum(1 for c in checks if c["token_available"] and not c["loose_value_match"]),
        }

    metadata["n_annotations_before_quality_filter"] = len(doc.annotations or [])
    metadata["n_annotations_after_quality_filter"] = len(kept)
    metadata["quality_filter_policy"] = {
        "min_token_overlap": MIN_TOKEN_OVERLAP,
        "drop_unavailable": drop_unavailable,
        "drop_loose_mismatch": drop_loose_mismatch,
        "implementation": "notebook_fast",
    }

    return doc_with_updates(doc, annotations=kept, metadata=metadata)

def fast_add_token_labels(doc, token_index, conflict_policy: str = "keep_first"):
    if doc.annotations is None:
        raise ValueError("Cannot add token labels: doc.annotations is None.")

    labels = ["O"] * len(doc.tokens)
    issues = []

    sorted_annotations = sorted(
        doc.annotations,
        key=lambda a: (int(a.start), int(a.end), str(a.label)),
    )

    for ann in sorted_annotations:
        indices = fast_tokens_for_annotation(
            doc.tokens,
            ann,
            token_index=token_index,
            min_token_overlap=MIN_TOKEN_OVERLAP,
        )

        if not indices:
            issues.append({
                "issue_type": "no_token_match",
                "label": ann.label,
                "start": int(ann.start),
                "end": int(ann.end),
                "text": ann.text,
            })
            continue

        encoded = label_to_key(ann.label)

        for j, idx in enumerate(indices):
            prefix = "B" if j == 0 else "I"
            new_label = f"{prefix}-{encoded}"

            if labels[idx] != "O":
                issues.append({
                    "issue_type": "label_conflict",
                    "old_label": labels[idx],
                    "new_label": new_label,
                    "token_index": idx,
                    "token_text": doc.tokens[idx].text,
                })

                if conflict_policy == "error":
                    raise ValueError(f"Label conflict at token {idx}")
                if conflict_policy == "keep_first":
                    continue

            labels[idx] = new_label

    metadata = dict(doc.metadata)
    metadata["token_labels"] = labels
    metadata["alignment_issues"] = issues

    return doc_with_updates(doc, metadata=metadata)

In [11]:
def maybe_drop_inline_ocr(df: pd.DataFrame) -> pd.DataFrame:
    if USE_INLINE_OCR_COLUMN:
        return df
    if "OCR" in df.columns:
        return df.drop(columns=["OCR"])
    return df

def short_error(e: Exception) -> str:
    return "".join(format_exception_only(type(e), e)).strip()

def should_validate(dataset_seen_index: int) -> bool:
    return VALIDATE_EVERY_RECORD or dataset_seen_index < VALIDATE_FIRST_N_DOCS_PER_DATASET

def should_assess_quality(dataset_seen_index: int) -> bool:
    return ASSESS_QUALITY_EVERY_RECORD or dataset_seen_index < ASSESS_QUALITY_FIRST_N_DOCS_PER_DATASET

def prepare_labeled_doc_fast(loader, row, assess_quality: bool):
    doc = loader.load_row(row, training=True)

    if not doc.annotations:
        return doc, None, None

    token_index = build_token_offset_index(doc.tokens)

    quality = None
    if assess_quality:
        quality = fast_quality_summary(doc, token_index)

    doc = fast_filter_annotations_for_training(
        doc,
        token_index=token_index,
        drop_unavailable=True,
        drop_loose_mismatch=False,
        attach_quality_report=assess_quality,
    )

    doc = fast_add_token_labels(
        doc,
        token_index=token_index,
        conflict_policy="keep_first",
    )

    return doc, doc.metadata["token_labels"], quality

def validate_serialized_train_record(record: dict, doc, token_labels: list[str]) -> None:
    n = len(record["tokens"])
    required_same_length = [
        "source_token_indices",
        "loss_mask",
        "layout_roles",
        "item_attrs",
        "labels",
        "pages",
        "bboxes",
        "normalized_bboxes",
        "offsets",
    ]

    for key in required_same_length:
        assert key in record, f"missing key: {key}"
        assert len(record[key]) == n, f"{record['serializer']}: {key} length mismatch"

    source_indices = record["source_token_indices"]
    loss_mask = record["loss_mask"]
    labels_out = record["labels"]

    real_indices = [idx for idx in source_indices if idx is not None]
    assert sorted(real_indices) == list(range(len(doc.tokens))), (
        f"{record['serializer']}: source_token_indices do not cover original tokens exactly once"
    )

    for pos, idx in enumerate(source_indices):
        if idx is None:
            assert loss_mask[pos] is False
            assert labels_out[pos] == IGNORE_LABEL
        else:
            assert loss_mask[pos] is True
            assert labels_out[pos] == token_labels[idx]

    collapsed = collapse_serialized_labels_to_original(record, labels_out)
    assert collapsed == token_labels, f"{record['serializer']}: BIO round-trip failed"

def record_stats(record: dict, doc, token_labels: list[str] | None) -> dict:
    real = sum(record["loss_mask"])
    layout = len(record["tokens"]) - real
    non_o = None
    if token_labels is not None:
        non_o = sum(1 for x in record.get("labels", []) if isinstance(x, str) and x != "O")

    return {
        "serialized_token_count": len(record["tokens"]),
        "original_token_count": len(doc.tokens),
        "real_token_count": real,
        "layout_token_count": layout,
        "length_multiplier": len(record["tokens"]) / max(1, len(doc.tokens)),
        "non_o_labels": non_o,
    }

def add_record_metadata(record, dataset_name, row, new_split, doc_key):
    record = dict(record)
    record["dataset"] = dataset_name
    record["split"] = new_split
    record["source_split"] = row["source_split"]
    record["source_folder"] = row["source_folder"]
    record["source_row_index"] = int(row["source_row_index"])
    record["combined_row_index"] = int(row["combined_row_index"])
    record["doc_key"] = doc_key

    record.setdefault("metadata", {})
    if isinstance(record["metadata"], dict):
        record["metadata"]["source_split"] = row["source_split"]
        record["metadata"]["new_seeded_split"] = new_split
        record["metadata"]["split_seed"] = SPLIT_SEED

    return record

def write_jsonl_line(handle, record: dict) -> None:
    handle.write(json.dumps(record, ensure_ascii=False) + "\n")

In [12]:
file_handles = {}

if WRITE_PROCESSED:
    PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)

    for dataset_name in combined_tables:
        for serializer_name in serializers:
            base_dir = PROCESSED_ROOT / dataset_name / serializer_name
            base_dir.mkdir(parents=True, exist_ok=True)

            file_handles[(dataset_name, serializer_name, "all")] = open(
                base_dir / "all.jsonl",
                "w",
                encoding="utf-8",
            )

            for split_name in ["train", "val", "test"]:
                file_handles[(dataset_name, serializer_name, split_name)] = open(
                    base_dir / f"{split_name}.jsonl",
                    "w",
                    encoding="utf-8",
                )

print("WRITE_PROCESSED:", WRITE_PROCESSED)
print("open file handles:", len(file_handles))

WRITE_PROCESSED: True
open file handles: 36


In [13]:
stats_rows = []
error_rows = []
timing_rows = []

try:
    for dataset_name, df in combined_tables.items():
        print(f"Processing {dataset_name}...")
        dataset_start = time.perf_counter()

        loader = loaders[dataset_name]
        work_df = maybe_drop_inline_ocr(df)

        for dataset_seen_index, (_, row) in enumerate(
            tqdm(work_df.iterrows(), total=len(work_df), desc=dataset_name)
        ):
            doc_key = make_doc_key(dataset_name, row)
            new_split = split_assignments[dataset_name][doc_key]

            try:
                prep_t0 = time.perf_counter()
                doc, token_labels, quality = prepare_labeled_doc_fast(
                    loader,
                    row,
                    assess_quality=should_assess_quality(dataset_seen_index),
                )
                prep_seconds = time.perf_counter() - prep_t0

                if token_labels is None:
                    raise ValueError("No annotations/token labels found.")

                for serializer_name, serializer in serializers.items():
                    t0 = time.perf_counter()
                    rec = serializer.serialize_train(doc)
                    serialize_seconds = time.perf_counter() - t0

                    rec = add_record_metadata(
                        rec,
                        dataset_name=dataset_name,
                        row=row,
                        new_split=new_split,
                        doc_key=doc_key,
                    )

                    if WRITE_PROCESSED:
                        write_jsonl_line(file_handles[(dataset_name, serializer_name, "all")], rec)
                        write_jsonl_line(file_handles[(dataset_name, serializer_name, new_split)], rec)


                    stats_rows.append({
                        "dataset": dataset_name,
                        "doc_id": doc.doc_id,
                        "doc_key": doc_key,
                        "source_split": row["source_split"],
                        "split": new_split,
                        "serializer": serializer_name,
                        "n_tokens": len(doc.tokens),
                        "n_annotations": len(doc.annotations or []),
                        "quality_token_available_rate": None if quality is None else quality["token_available_rate"],
                        "quality_strict_match_rate": None if quality is None else quality["strict_match_rate"],
                        "quality_loose_value_match_rate": None if quality is None else quality["loose_value_match_rate"],
                        "prep_seconds": prep_seconds,
                        "serialize_seconds": serialize_seconds,
                        **record_stats(rec, doc, token_labels),
                    })

                    timing_rows.append({
                        "dataset": dataset_name,
                        "serializer": serializer_name,
                        "doc_key": doc_key,
                        "prep_seconds": prep_seconds,
                        "serialize_seconds": serialize_seconds,
                        "serialized_token_count": len(rec["tokens"]),
                    })

            except Exception as e:
                error_rows.append({
                    "dataset": dataset_name,
                    "doc_key": doc_key,
                    "source_split": row.get("source_split"),
                    "assigned_split": new_split,
                    "error": short_error(e),
                })

        print(f"Finished {dataset_name} in {time.perf_counter() - dataset_start:.2f}s")

finally:
    for handle in file_handles.values():
        handle.flush()
        handle.close()

stats_df = pd.DataFrame(stats_rows)
errors_df = pd.DataFrame(error_rows)
timing_df = pd.DataFrame(timing_rows)

print("stats rows:", len(stats_df))
print("errors:", len(errors_df))

display(stats_df.head())
display(errors_df.head(50))

Processing resource_contracts...


resource_contracts:   0%|          | 0/198 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
rec

## 11. Timing diagnostics

In [ ]:
if timing_df.empty:
    print("No timing data.")
else:
    prep_summary = (
        timing_df.drop_duplicates(["dataset", "doc_key"])
        .groupby("dataset")
        .agg(
            n_docs=("doc_key", "nunique"),
            mean_prep_seconds=("prep_seconds", "mean"),
            max_prep_seconds=("prep_seconds", "max"),
        )
        .reset_index()
        .sort_values("mean_prep_seconds", ascending=False)
    )
    display(prep_summary)

    serializer_timing = (
        timing_df.groupby(["dataset", "serializer"])
        .agg(
            n_docs=("doc_key", "nunique"),
            mean_serialize_seconds=("serialize_seconds", "mean"),
            p95_serialize_seconds=("serialize_seconds", lambda x: x.quantile(0.95)),
            max_serialize_seconds=("serialize_seconds", "max"),
            mean_tokens=("serialized_token_count", "mean"),
            max_tokens=("serialized_token_count", "max"),
        )
        .reset_index()
        .sort_values(["dataset", "mean_serialize_seconds"], ascending=[True, False])
    )
    display(serializer_timing)

    display(timing_df.sort_values("serialize_seconds", ascending=False).head(25))

## 12. Split and length summaries

In [ ]:
if stats_df.empty:
    print("No stats available.")
else:
    split_dist = (
        stats_df.drop_duplicates(["dataset", "doc_key"])
        .groupby(["dataset", "split"])
        .size()
        .reset_index(name="n_docs")
    )
    display(split_dist.pivot(index="dataset", columns="split", values="n_docs").fillna(0).astype(int))

    length_summary = (
        stats_df.groupby(["dataset", "serializer"])
        .agg(
            n_docs=("doc_key", "nunique"),
            mean_original_tokens=("original_token_count", "mean"),
            mean_serialized_tokens=("serialized_token_count", "mean"),
            max_serialized_tokens=("serialized_token_count", "max"),
            mean_layout_tokens=("layout_token_count", "mean"),
            mean_length_multiplier=("length_multiplier", "mean"),
        )
        .reset_index()
        .sort_values(["dataset", "mean_length_multiplier", "serializer"])
    )
    display(length_summary)

## 13. Length-limit stress test

In [ ]:
MAX_LENGTHS = [512, 1024, 2048, 4096, 8192, 16384, 32768]

if stats_df.empty:
    print("No stats available.")
else:
    limit_rows = []
    doc_level = stats_df.drop_duplicates(["dataset", "doc_key", "serializer"])

    for (dataset_name, serializer_name), group in doc_level.groupby(["dataset", "serializer"]):
        for max_len in MAX_LENGTHS:
            limit_rows.append({
                "dataset": dataset_name,
                "serializer": serializer_name,
                "max_length": max_len,
                "pct_over": float((group["serialized_token_count"] > max_len).mean()),
            })

    limit_df = pd.DataFrame(limit_rows)

    for dataset_name in sorted(limit_df["dataset"].unique()):
        print("\nDataset:", dataset_name)
        display(
            limit_df[limit_df["dataset"] == dataset_name]
            .pivot(index="serializer", columns="max_length", values="pct_over")
            .sort_index()
        )

## 14. Inspect one preview record

In [ ]:
loaded_datasets = set(combined_tables.keys())
successful_datasets = set(stats_df["dataset"].unique()) if not stats_df.empty else set()
required_serializer_names = set(serializers.keys())

if not stats_df.empty:
    observed_pairs = set(zip(stats_df["dataset"], stats_df["serializer"]))
    expected_pairs = {
        (dataset_name, serializer_name)
        for dataset_name in successful_datasets
        for serializer_name in required_serializer_names
    }
    split_values = set(stats_df["split"].unique())
else:
    observed_pairs = set()
    expected_pairs = set()
    split_values = set()

checks = {
    "at_least_one_dataset_successful": bool(successful_datasets),
    "all_loaded_datasets_successful": loaded_datasets == successful_datasets,
    "all_selected_serializers_successful": expected_pairs.issubset(observed_pairs),
    "seeded_splits_present": {"train", "val", "test"}.issubset(split_values),
    "no_errors": errors_df.empty,
}

checks["overall_pass_strict"] = all(checks.values())
checks["overall_pass_functional"] = (
    checks["at_least_one_dataset_successful"]
    and checks["all_selected_serializers_successful"]
    and checks["seeded_splits_present"]
)

print(json.dumps({
    "run_mode": RUN_MODE,
    "split_seed": SPLIT_SEED,
    "include_heavy_serializers": INCLUDE_HEAVY_SERIALIZERS,
    "loaded_datasets": sorted(loaded_datasets),
    "successful_datasets": sorted(successful_datasets),
    "selected_serializers": sorted(required_serializer_names),
    "n_errors": int(len(errors_df)),
    "checks": checks,
}, indent=2))

if checks["overall_pass_strict"]:
    print("\nSTRICT PASS.")
elif checks["overall_pass_functional"]:
    print("\nFUNCTIONAL PASS. Inspect errors if any.")
else:
    print("\nFAIL/WARN.")